In [ ]:
# ============================================================
# CELL 1 - Environment Setup
# Install all required dependencies for MMSegmentation + SegFormer
# ============================================================

# Install PyTorch with CUDA 12.1 support
!pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Install OpenMMLab stack
!pip install mmengine -q
!pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html -q
!pip install mmsegmentation -q

# Fix mmcv version check (mmseg 1.2.2 hardcodes max version as 2.2.0 exclusive)
!sed -i 's/MMCV_MAX = .2.2.0./MMCV_MAX = "2.3.0"/' \
    /usr/local/lib/python3.12/dist-packages/mmseg/__init__.py

print("Installation complete, version check fixed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 842.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 62.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 115.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 750.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL 2 - Environment Verification
# Verify all packages are correctly installed and GPU is available
# ============================================================

import torch, mmcv, mmengine, mmseg

print('torch:', torch.__version__)
print('mmcv:', mmcv.__version__)
print('mmseg:', mmseg.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.4.1+cu121
mmcv: 2.2.0
mmseg: 1.2.2
CUDA available: True
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 3 - Mount Google Drive and Extract Dataset
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Extract dataset from Drive to Colab local storage (faster I/O during training)
!unzip -q /content/drive/MyDrive/Dataset/processed.zip -d /content/

# Verify dataset structure and image counts
print("=== Dataset Verification ===")
!echo "Train images:" && ls /content/processed/train/images/ | wc -l
!echo "Train masks: " && ls /content/processed/train/masks/  | wc -l
!echo "Val images:  " && ls /content/processed/val/images/   | wc -l
!echo "Val masks:   " && ls /content/processed/val/masks/    | wc -l
!echo "Test images: " && ls /content/processed/test/images/  | wc -l
!echo "Test masks:  " && ls /content/processed/test/masks/   | wc -l

Mounted at /content/drive
=== Dataset Verification ===
Train images:
672
Train masks: 
672
Val images:  
149
Val masks:   
149
Test images: 
146
Test masks:  
146


In [15]:
# ============================================================
# CELL 4 - Clone Repository and Download Pretrained Weights
# ============================================================

import os

# Clone the project repository
!git clone https://github.com/ilMassy/advertising-panel-segmentation /content/repo
%cd /content/repo

# Create checkpoints directory (models/ and results/ already exist in the repo)
os.makedirs('checkpoints', exist_ok=True)

# Download MIT-B0 pretrained weights (for baseline experiment)
!wget -q --show-progress \
    https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth \
    -O checkpoints/mit_b0_imagenet.pth

# Download MIT-B1 pretrained weights (for main experiment)
!wget -q --show-progress \
    https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b1_20220624-02e5a6a1.pth \
    -O checkpoints/mit_b1_imagenet.pth

# Verify downloads
!echo "Checkpoint sizes:"
!ls -lh checkpoints/

fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo
checkpoints/mit_b0_ 100%[===================>]  12.71M  9.75MB/s    in 1.3s    
checkpoints/mit_b1_ 100%[===================>]  50.22M  15.9MB/s    in 3.2s    
Checkpoint sizes:
total 63M
-rw-r--r-- 1 root root 13M Jun 24  2022 mit_b0_imagenet.pth
-rw-r--r-- 1 root root 51M Jun 24  2022 mit_b1_imagenet.pth


In [16]:
# ============================================================
# CELL 5 - Verify Repository Structure
# All scripts and configs are cloned from GitHub
# ============================================================

import os

# Verify all files were cloned correctly
!echo "=== Config files ===" && ls /content/repo/configs/
!echo "=== Source scripts ===" && ls /content/repo/src/
!echo "=== Results directory ===" && ls /content/repo/results/
!echo "=== Models directory ===" && ls /content/repo/models/

# Override config files with fixed optimizer (removes momentum inheritance from base config)
os.makedirs('/content/repo/configs', exist_ok=True)

segformer_b0_config = """
_base_ = [
    'mmseg::_base_/models/segformer_mit-b0.py',
    'mmseg::_base_/default_runtime.py',
]

# Dataset settings
dataset_type = 'BaseSegDataset'
data_root    = '/content/processed/'
crop_size    = (640, 640)

# Training pipeline
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(1920, 1080), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PhotoMetricDistortion'),
    dict(type='PackSegInputs'),
]

# Validation pipeline
val_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(1920, 1080), keep_ratio=True),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

# Dataset meta info (2 classes: background and board)
meta = dict(
    classes=('background', 'board'),
    palette=[[0, 0, 0], [255, 0, 0]]
)

# Dataloaders
train_dataloader = dict(
    batch_size=4, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=train_pipeline,
    )
)

val_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='val/images', seg_map_path='val/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

test_dataloader = val_dataloader

# Evaluation metrics
val_evaluator  = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice'])
test_evaluator = val_evaluator

# Model: 2 classes with pretrained backbone
model = dict(
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=True,
        pad_val=0,
        seg_pad_val=255,
        size=crop_size,
        size_divisor=None,
    ),
    backbone=dict(
        init_cfg=dict(
            type='Pretrained',
            checkpoint='/content/repo/checkpoints/mit_b0_imagenet.pth'
        )
    ),
    decode_head=dict(num_classes=2),
    test_cfg=dict(mode='whole')
)

# AdamW optimizer - explicitly override base config to prevent momentum inheritance
optimizer = dict(type='AdamW', lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01)
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=optimizer,
    paramwise_cfg=dict(
        custom_keys={
            'pos_block': dict(decay_mult=0.),
            'norm':      dict(decay_mult=0.),
            'head':      dict(lr_mult=10.)
        }
    )
)

# LR scheduler with warmup
param_scheduler = [
    dict(type='LinearLR', start_factor=1e-6, by_epoch=False, begin=0,    end=1500),
    dict(type='PolyLR',   eta_min=0.0,       power=1.0,      begin=1500, end=160000, by_epoch=False),
]

# Save best checkpoint based on mIoU
default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', by_epoch=False, interval=2000,
                    max_keep_ckpts=3, save_best='mIoU', rule='greater'),
    logger=dict(type='LoggerHook', interval=50),
)

train_cfg = dict(type='IterBasedTrainLoop', max_iters=20000, val_interval=2000)
val_cfg   = dict(type='ValLoop')
test_cfg  = dict(type='TestLoop')
"""

# B1 config: written from scratch with correct B1 architecture
segformer_b1_config = """
_base_ = [
    'mmseg::_base_/models/segformer_mit-b0.py',
    'mmseg::_base_/default_runtime.py',
]

# Dataset settings
dataset_type = 'BaseSegDataset'
data_root    = '/content/processed/'
crop_size    = (640, 640)

# Training pipeline
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(1920, 1080), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PhotoMetricDistortion'),
    dict(type='PackSegInputs'),
]

# Validation pipeline
val_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(1920, 1080), keep_ratio=True),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

# Dataset meta info (2 classes: background and board)
meta = dict(
    classes=('background', 'board'),
    palette=[[0, 0, 0], [255, 0, 0]]
)

# Dataloaders
train_dataloader = dict(
    batch_size=4, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=train_pipeline,
    )
)

val_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='val/images', seg_map_path='val/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

test_dataloader = val_dataloader

# Evaluation metrics
val_evaluator  = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice'])
test_evaluator = val_evaluator

# Model: SegFormer-B1 architecture (larger than B0)
# B1: embed_dims=64, vs B0: embed_dims=32
model = dict(
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=True,
        pad_val=0,
        seg_pad_val=255,
        size=crop_size,
        size_divisor=None,
    ),
    backbone=dict(
        embed_dims=64,
        num_heads=[1, 2, 5, 8],
        num_layers=[2, 2, 2, 2],
        init_cfg=dict(
            type='Pretrained',
            checkpoint='/content/repo/checkpoints/mit_b1_imagenet.pth'
        )
    ),
    decode_head=dict(
        in_channels=[64, 128, 320, 512],
        num_classes=2,
    ),
    test_cfg=dict(mode='whole')
)

# AdamW optimizer - explicitly override base config to prevent momentum inheritance
optimizer = dict(type='AdamW', lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01)
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=optimizer,
    paramwise_cfg=dict(
        custom_keys={
            'pos_block': dict(decay_mult=0.),
            'norm':      dict(decay_mult=0.),
            'head':      dict(lr_mult=10.)
        }
    )
)

# LR scheduler with warmup
param_scheduler = [
    dict(type='LinearLR', start_factor=1e-6, by_epoch=False, begin=0,    end=1500),
    dict(type='PolyLR',   eta_min=0.0,       power=1.0,      begin=1500, end=160000, by_epoch=False),
]

# Save best checkpoint based on mIoU
default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', by_epoch=False, interval=2000,
                    max_keep_ckpts=3, save_best='mIoU', rule='greater'),
    logger=dict(type='LoggerHook', interval=50),
)

train_cfg = dict(type='IterBasedTrainLoop', max_iters=20000, val_interval=2000)
val_cfg   = dict(type='ValLoop')
test_cfg  = dict(type='TestLoop')
"""

# Write fixed config files to disk (overrides cloned versions)
with open('/content/repo/configs/segformer_b0_baseline.py', 'w') as f:
    f.write(segformer_b0_config)

with open('/content/repo/configs/segformer_b1_standard.py', 'w') as f:
    f.write(segformer_b1_config)

print("Config files verified and fixed!")
!ls -lh /content/repo/configs/

=== Config files ===
segformer_b0_baseline.py  segformer_b1_standard.py
=== Source scripts ===
check_masks.py	CVAT_preparation.py  extract_frames.py	reorder_by_prefix.py
=== Results directory ===
exp0_segformer_b0_baseline  exp1_segformer_b1_standard
=== Models directory ===
README.md
Config files verified and fixed!
total 8.0K
-rw-r--r-- 1 root root 3.3K Mar 23 21:55 segformer_b0_baseline.py
-rw-r--r-- 1 root root 3.5K Mar 23 21:55 segformer_b1_standard.py


In [ ]:
# ============================================================
# CELL 6 - Dataset Verification
# Quick sanity check before launching training
# ============================================================

!pip install ftfy regex -q

from mmengine.config import Config
from mmseg.datasets import BaseSegDataset

# Load config
cfg = Config.fromfile('configs/segformer_b0_baseline.py')

# Test dataset loading
dataset = BaseSegDataset(
    data_root='/content/processed/',
    data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
    img_suffix='.jpeg',
    seg_map_suffix='.png',
    metainfo=dict(classes=('background', 'board'), palette=[[0,0,0],[255,0,0]])
)

print(f'Train dataset size: {len(dataset)}')
print(f'Expected: 672')

# Check one sample
sample = dataset[0]
print(f'Sample keys: {sample.keys()}')
print('Dataset verification OK!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


Train dataset size: 672
Expected: 672
Sample keys: dict_keys(['img_path', 'seg_map_path', 'label_map', 'reduce_zero_label', 'seg_fields', 'sample_idx'])
Dataset verification OK!


In [ ]:
# ============================================================
# CELL 7 - Push Config Files to GitHub
# Save MMSegmentation config files to the repository
# ============================================================

import os
from getpass import getpass

# Git configuration
!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"

# Ask for token securely (input is hidden)
GITHUB_TOKEN = getpass("Enter your GitHub Personal Access Token: ")

# Set remote URL with authentication token
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git

# Pull remote changes first
!cd /content/repo && git pull origin main --rebase

# Add and commit config files
!cd /content/repo && git add configs/
!cd /content/repo && git commit -m "Add MMSegmentation config files for B0 baseline and B1 standard" \
    || echo "Nothing to commit, already done"

# Push to GitHub
!cd /content/repo && git push origin main

print("Config files pushed to GitHub!")

Enter your GitHub Personal Access Token: ··········
From https://github.com/ilMassy/advertising-panel-segmentation
 * branch            main       -> FETCH_HEAD
Already up to date.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Nothing to commit, already done
Everything up-to-date
Config files pushed to GitHub!


In [ ]:
# ============================================================
# CELL 8 - Training: SegFormer-B0 Baseline (Phase 2)
# Establishes the reference IoU benchmark
# ============================================================

import os
import mmseg

# Correct path for mmseg tools on Colab
mmseg_tools = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'train.py'
)
print(f"MMSeg train tool: {mmseg_tools}")

# Verify tool exists
assert os.path.exists(mmseg_tools), f"train.py not found at {mmseg_tools}"
print("Train tool found!")

# Launch B0 baseline training (checkpoint already specified in config)
!python {mmseg_tools} \
    /content/repo/configs/segformer_b0_baseline.py \
    --work-dir /content/repo/results/exp0_segformer_b0_baseline

MMSeg train tool: /usr/local/lib/python3.12/dist-packages/mmseg/.mim/tools/train.py
Train tool found!
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/21 21:59:28 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 437083542
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Inte

In [ ]:
# ============================================================
# CELL 9 - Save B0 Results to Google Drive
# Backup training results before Colab disconnects
# ============================================================

import os

# Create all necessary folders on Google Drive
!mkdir -p /content/drive/MyDrive/advertising_panel_segmentation/results

# Copy B0 results to Google Drive (persistent storage)
!cp -r /content/repo/results/exp0_segformer_b0_baseline \
    /content/drive/MyDrive/advertising_panel_segmentation/results/

# Verify
print("Results saved to Google Drive!")
!ls /content/drive/MyDrive/advertising_panel_segmentation/results/

Results saved to Google Drive!
exp0_segformer_b0_baseline


In [8]:
# ============================================================
# CELL 10 - Training: SegFormer-B1 Standard (Phase 3)
# Fine-tuning B1 on custom dataset, comparison against B0 baseline
# Checkpoints saved directly to Google Drive during training
# ============================================================

import os
import mmseg

# Correct path for mmseg tools on Colab (same as B0)
mmseg_tools = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'train.py'
)
print(f"MMSeg train tool: {mmseg_tools}")

# Verify tool exists
assert os.path.exists(mmseg_tools), f"train.py not found at {mmseg_tools}"
print("Train tool found!")

# Launch B1 standard training
# work-dir points directly to Drive so checkpoints are saved during training
!python {mmseg_tools} \
    /content/repo/configs/segformer_b1_standard.py \
    --work-dir /content/repo/results/exp1_segformer_b1_standard

MMSeg train tool: /usr/local/lib/python3.12/dist-packages/mmseg/.mim/tools/train.py
Train tool found!
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/23 17:47:55 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 155641638
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Inte

In [9]:
# ============================================================
# CELL 11 - Save B1 Results to Google Drive
# Backup training results before Colab disconnects
# ============================================================

import os

# Create all necessary folders on Google Drive (if not already existing)
!mkdir -p /content/drive/MyDrive/advertising_panel_segmentation/results

# Copy B1 results to Google Drive (persistent storage)
!cp -r /content/repo/results/exp1_segformer_b1_standard \
    /content/drive/MyDrive/advertising_panel_segmentation/results/

# Verify
print("Results saved to Google Drive!")
!ls /content/drive/MyDrive/advertising_panel_segmentation/results/

Results saved to Google Drive!
exp0_segformer_b0_baseline  exp1_segformer_b1_standard


In [11]:
# ============================================================
# CELL 12 - Evaluate B0 and B1 with Full Metrics
# Precision and Recall included
# Uses best checkpoints saved on Google Drive
# ============================================================

import mmseg, os, glob

mmseg_test = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'test.py'
)

def evaluate_model(exp_name, config_path):
    print(f"\n{'='*60}")
    print(f"Evaluating: {exp_name}")
    print(f"{'='*60}")

    # Automatically find best checkpoint on Drive
    checkpoints = glob.glob(
        f'/content/drive/MyDrive/advertising_panel_segmentation/results/{exp_name}/best_mIoU_iter_*.pth'
    )

    if not checkpoints:
        print(f"No checkpoint found for {exp_name}! Run Cell 9 and 11 first.")
        return

    # Pick checkpoint with highest iteration number
    best_checkpoint = max(
        checkpoints,
        key=lambda x: int(x.split('iter_')[1].split('.')[0])
    )
    print(f"Using checkpoint: {best_checkpoint}")

    # Run evaluation with full metrics including Precision and Recall
    # Must override test_evaluator (not val_evaluator) for test.py
    !python {mmseg_test} \
        {config_path} \
        {best_checkpoint} \
        --cfg-options \
            test_evaluator.iou_metrics="['mIoU','mDice','mFscore']"

# Evaluate B0 baseline
evaluate_model(
    exp_name='exp0_segformer_b0_baseline',
    config_path='/content/repo/configs/segformer_b0_baseline.py'
)

# Evaluate B1 standard
evaluate_model(
    exp_name='exp1_segformer_b1_standard',
    config_path='/content/repo/configs/segformer_b1_standard.py'
)


Evaluating: exp0_segformer_b0_baseline
Using checkpoint: /content/drive/MyDrive/advertising_panel_segmentation/results/exp0_segformer_b0_baseline/best_mIoU_iter_18000.pth
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/23 21:20:01 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1722224483
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling de

In [33]:
# ============================================================
# CELL 13 - Save Logs and Model Summary to Repository
# Copies training logs from Drive to GitHub repo
# ============================================================

import os, glob, shutil

# ── Create directories ────────────────────────────────────────────────────────
os.makedirs('/content/repo/results/exp0_segformer_b0_baseline', exist_ok=True)
os.makedirs('/content/repo/results/exp1_segformer_b1_standard', exist_ok=True)
os.makedirs('/content/repo/models', exist_ok=True)

# ── Copy training logs from Drive to results/ ─────────────────────────────────
for exp in ['exp0_segformer_b0_baseline', 'exp1_segformer_b1_standard']:
    src = f'/content/drive/MyDrive/advertising_panel_segmentation/results/{exp}'
    dst = f'/content/repo/results/{exp}'

    # Copy log files only (lightweight, no .pth weights)
    log_files = glob.glob(f'{src}/**/*.log', recursive=True)
    if log_files:
        for log_file in log_files:
            shutil.copy(log_file, dst)
            print(f"Copied: {os.path.basename(log_file)} → results/{exp}/")
    else:
        print(f"No log files found in {src}")

# ── Create model summary in models/ ──────────────────────────────────────────
model_summary = """# Trained Models Summary

## Exp0 - SegFormer-B0 Baseline
- Best checkpoint: best_mIoU_iter_18000.pth
- mIoU: 87.77%
- board IoU: 78.09%
- Dice: 87.70%
- Precision: 83.15%
- Recall: 92.77%
- Stored on: Google Drive

## Exp1 - SegFormer-B1 Standard
- Best checkpoint: best_mIoU_iter_6000.pth
- mIoU: 88.90%
- board IoU: 80.01%
- Dice: 88.89%
- Precision: 87.67%
- Recall: 90.15%
- Stored on: Google Drive
"""

with open('/content/repo/models/README.md', 'w') as f:
    f.write(model_summary)
print("Created models/README.md")

# ── Verify only logs are present (no .pth) ───────────────────────────────────
print("\n=== results/ ===")
!find /content/repo/results -type f | grep -v __pycache__
print("\n=== models/ ===")
!ls /content/repo/models/

# ── Verify no .pth files exist ───────────────────────────────────────────────
pth_files = glob.glob('/content/repo/results/**/*.pth', recursive=True)
if pth_files:
    print(f"WARNING: .pth files found! Removing...")
    for f in pth_files:
        os.remove(f)
        print(f"Removed: {f}")
else:
    print("No .pth files found - safe to push!")

# ── Push to GitHub (only logs and README) ────────────────────────────────────
from getpass import getpass
GITHUB_TOKEN = getpass("Enter GitHub token: ")

!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git
!cd /content/repo && git pull origin main --rebase

# Add only specific files (not the entire results/ folder)
!cd /content/repo && git add results/exp0_segformer_b0_baseline/*.log
!cd /content/repo && git add results/exp1_segformer_b1_standard/*.log
!cd /content/repo && git add models/README.md

# Verify what will be committed
print("\n=== Files to be committed ===")
!cd /content/repo && git diff --cached --name-only

!cd /content/repo && git commit -m "Add training logs and model summary for B0 and B1" \
    || echo "Nothing to commit"
!cd /content/repo && git push origin main
print("Pushed to GitHub!")

Copied: 20260321_214403.log → results/exp0_segformer_b0_baseline/
Copied: 20260321_214728.log → results/exp0_segformer_b0_baseline/
Copied: 20260321_215047.log → results/exp0_segformer_b0_baseline/
Copied: 20260321_215248.log → results/exp0_segformer_b0_baseline/
Copied: 20260321_215435.log → results/exp0_segformer_b0_baseline/
Copied: 20260321_215927.log → results/exp0_segformer_b0_baseline/
Copied: 20260323_174754.log → results/exp1_segformer_b1_standard/
Created models/README.md

=== results/ ===
/content/repo/results/exp1_segformer_b1_standard/20260323_174754/20260323_174754.log
/content/repo/results/exp1_segformer_b1_standard/20260323_174754/vis_data/scalars.json
/content/repo/results/exp1_segformer_b1_standard/20260323_174754/vis_data/config.py
/content/repo/results/exp1_segformer_b1_standard/20260323_174754/vis_data/20260323_174754.json
/content/repo/results/exp1_segformer_b1_standard/20260323_174754.log
/content/repo/results/exp0_segformer_b0_baseline/20260321_215047.log
/conte